In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../0_common/04.gold-helpers"

In [0]:
%run "../0_common/01. env-config"

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
drivers_df = spark.table(f"{catalog_name}.{silver_schema}.drivers").filter((F.col("batch_id") == v_batch_id))
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
dim_drivers_df = (
    drivers_df
    .join(ref_nationality_region_df, on='nationality', how='left')
    .select(
        'driver_id',
        'driver_name',
        'date_of_birth',
        'nationality',
        ref_nationality_region_df.region.alias('nationality_region')
    )
)

In [0]:
columns_to_update = [col for col in dim_drivers_df.columns if col not in ["driver_id"]]
columns_to_update

In [0]:
write_to_gold(
    df=dim_drivers_df,
    target_table=target_table,
    columns_to_update=columns_to_update,
    merge_condition="t.driver_id=s.driver_id"
)

In [0]:
display(spark.table(target_table))